# Class Imbalance Handling
## Fraud Detection Project - Task 1b

This notebook covers:
- Class distribution analysis
- Applying SMOTE for oversampling
- Before/after comparison
- Preparing balanced data for modeling

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('../src')

from preprocessor import Preprocessor
from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
%matplotlib inline

## 1. Load Engineered Data

In [ ]:
# Load feature-engineered data
fraud_df = pd.read_csv('../data/processed/fraud_data_engineered.csv')
print(f"Data shape: {fraud_df.shape}")
fraud_df.head()

## 2. Class Distribution Before Resampling

In [ ]:
# Class distribution BEFORE resampling
class_dist_before = fraud_df['class'].value_counts()
class_pct_before = fraud_df['class'].value_counts(normalize=True) * 100

print("="*50)
print("CLASS DISTRIBUTION BEFORE RESAMPLING")
print("="*50)
print(f"\nNon-Fraud (0): {class_dist_before[0]:,} ({class_pct_before[0]:.2f}%)")
print(f"Fraud (1):     {class_dist_before[1]:,} ({class_pct_before[1]:.2f}%)")
print(f"\nImbalance Ratio: 1:{class_dist_before[0]/class_dist_before[1]:.2f}")
print(f"Total Samples: {len(fraud_df):,}")

In [ ]:
# Visualize class distribution before resampling
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
colors = ['#2ecc71', '#e74c3c']
class_dist_before.plot(kind='bar', ax=axes[0], color=colors)
axes[0].set_title('Class Distribution BEFORE Resampling', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['Non-Fraud (0)', 'Fraud (1)'], rotation=0)

# Add value labels
for i, v in enumerate(class_dist_before):
    axes[0].text(i, v + 1000, f'{v:,}', ha='center', fontweight='bold')

# Pie chart
axes[1].pie(class_dist_before, labels=['Non-Fraud', 'Fraud'], 
            autopct='%1.1f%%', colors=colors, startangle=90,
            explode=(0, 0.1))
axes[1].set_title('Class Distribution BEFORE Resampling', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../docs/class_dist_before.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Prepare Data for SMOTE

In [ ]:
# Select features for modeling
# Remove non-numeric and unnecessary columns
cols_to_drop = ['user_id', 'signup_time', 'purchase_time', 'device_id', 
                'ip_address', 'country', 'time_of_day', 'purchase_date',
                'source', 'browser', 'sex']  # Original categorical columns

# Keep only columns that exist
cols_to_drop = [col for col in cols_to_drop if col in fraud_df.columns]

# Prepare features
X = fraud_df.drop(['class'] + cols_to_drop, axis=1)
y = fraud_df['class']

# Ensure all numeric
X = X.select_dtypes(include=[np.number])

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature columns: {X.columns.tolist()}")

In [ ]:
# Handle any remaining NaN values
if X.isnull().sum().sum() > 0:
    print(f"Found {X.isnull().sum().sum()} NaN values, filling with median...")
    X = X.fillna(X.median())
else:
    print("No NaN values found.")

## 4. Apply SMOTE

In [ ]:
# Initialize preprocessor
preprocessor = Preprocessor()

# Apply SMOTE
print("Applying SMOTE...")
X_resampled, y_resampled = preprocessor.handle_class_imbalance(X, y, sampling_strategy=1.0)

print(f"\nOriginal shape: {X.shape}")
print(f"Resampled shape: {X_resampled.shape}")

## 5. Class Distribution After Resampling

In [ ]:
# Class distribution AFTER resampling
y_resampled_series = pd.Series(y_resampled)
class_dist_after = y_resampled_series.value_counts()
class_pct_after = y_resampled_series.value_counts(normalize=True) * 100

print("="*50)
print("CLASS DISTRIBUTION AFTER RESAMPLING (SMOTE)")
print("="*50)
print(f"\nNon-Fraud (0): {class_dist_after[0]:,} ({class_pct_after[0]:.2f}%)")
print(f"Fraud (1):     {class_dist_after[1]:,} ({class_pct_after[1]:.2f}%)")
print(f"\nBalance Ratio: 1:{class_dist_after[0]/class_dist_after[1]:.2f}")
print(f"Total Samples: {len(y_resampled):,}")
print(f"\nNew synthetic samples created: {len(y_resampled) - len(y):,}")

In [ ]:
# Visualize class distribution after resampling
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
class_dist_after.plot(kind='bar', ax=axes[0], color=colors)
axes[0].set_title('Class Distribution AFTER Resampling (SMOTE)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['Non-Fraud (0)', 'Fraud (1)'], rotation=0)

# Add value labels
for i, v in enumerate(class_dist_after):
    axes[0].text(i, v + 1000, f'{v:,}', ha='center', fontweight='bold')

# Pie chart
axes[1].pie(class_dist_after, labels=['Non-Fraud', 'Fraud'], 
            autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('Class Distribution AFTER Resampling', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../docs/class_dist_after.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Before/After Comparison

In [ ]:
# Side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before
class_dist_before.plot(kind='bar', ax=axes[0], color=colors)
axes[0].set_title('BEFORE Resampling', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['Non-Fraud', 'Fraud'], rotation=0)
for i, v in enumerate(class_dist_before):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', fontsize=10)

# After
class_dist_after.plot(kind='bar', ax=axes[1], color=colors)
axes[1].set_title('AFTER Resampling (SMOTE)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Count')
axes[1].set_xticklabels(['Non-Fraud', 'Fraud'], rotation=0)
for i, v in enumerate(class_dist_after):
    axes[1].text(i, v + 500, f'{v:,}', ha='center', fontsize=10)

plt.suptitle('Class Imbalance Handling with SMOTE', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../docs/class_dist_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Comparison table
comparison_df = pd.DataFrame({
    'Metric': ['Non-Fraud Count', 'Fraud Count', 'Total Samples', 'Fraud Percentage', 'Imbalance Ratio'],
    'Before SMOTE': [
        f"{class_dist_before[0]:,}",
        f"{class_dist_before[1]:,}",
        f"{len(fraud_df):,}",
        f"{class_pct_before[1]:.2f}%",
        f"1:{class_dist_before[0]/class_dist_before[1]:.2f}"
    ],
    'After SMOTE': [
        f"{class_dist_after[0]:,}",
        f"{class_dist_after[1]:,}",
        f"{len(y_resampled):,}",
        f"{class_pct_after[1]:.2f}%",
        f"1:{class_dist_after[0]/class_dist_after[1]:.2f}"
    ]
})

print("\n" + "="*60)
print("CLASS DISTRIBUTION COMPARISON")
print("="*60)
print(comparison_df.to_string(index=False))

## 7. Save Balanced Data

In [ ]:
# Create balanced DataFrame
balanced_df = pd.DataFrame(X_resampled, columns=X.columns)
balanced_df['class'] = y_resampled

# Save
balanced_df.to_csv('../data/processed/balanced_fraud_data.csv', index=False)
print(f"Balanced data saved: {balanced_df.shape}")

In [ ]:
# Preprocessing report
print(preprocessor.get_preprocessing_report())

## Summary

### Class Imbalance Handling Complete!

**Method Used**: SMOTE (Synthetic Minority Over-sampling Technique)

**Results**:
- Successfully balanced the dataset from highly imbalanced to 1:1 ratio
- Created synthetic samples for the minority class (Fraud)
- Preserved original majority class samples

**Output Files**:
- `../data/processed/balanced_fraud_data.csv` - Ready for model training

**Next Steps**:
- Model training with balanced dataset
- Cross-validation and evaluation
- Compare with other imbalance handling techniques